# FRED data cleaning
Merge six FRED series (daily, weekly, monthly, quarterly) into one monthly table for the Power BI dashboard.

Output: `data/fred_clean.csv`

In [1]:
from pathlib import Path
import pandas as pd

DATA = Path("data")
SERIES = ["MORTGAGE30US", "FEDFUNDS", "DGS10", "UNRATE", "CPIAUCSL", "DRSFRMACBS"]
START = "1991-01-01"  # delinquency series begins in 1991

## 1. Load each raw file
FRED writes `.` for missing values, so convert to numeric and count the nulls.

In [2]:
def load(sid):
    df = pd.read_csv(DATA / f"{sid}.csv", parse_dates=["observation_date"])
    s = pd.to_numeric(df[sid], errors="coerce")
    print(f"{sid:<13} rows={len(df):>6}  missing={s.isna().sum():>3}  "
          f"{df.observation_date.min():%Y-%m-%d} -> {df.observation_date.max():%Y-%m-%d}")
    return pd.Series(s.values, index=df.observation_date, name=sid)

raw = {sid: load(sid) for sid in SERIES}

MORTGAGE30US  rows=  2896  missing=  0  1971-04-02 -> 2026-09-24
FEDFUNDS      rows=   866  missing=  0  1954-07-01 -> 2026-08-01
DGS10         rows= 16890  missing=720  1962-01-02 -> 2026-09-28
UNRATE        rows=   944  missing=  1  1948-01-01 -> 2026-08-01
CPIAUCSL      rows=   956  missing=  1  1947-01-01 -> 2026-08-01
DRSFRMACBS    rows=   142  missing=  0  1991-01-01 -> 2026-04-01


## 2. Convert everything to monthly
Daily (DGS10) and weekly (MORTGAGE30US) series become monthly averages. Monthly series pass through. The quarterly delinquency rate is carried across the 3 months of its quarter. Month-start dates (`MS`) are used so Power BI treats them as dates cleanly.

In [3]:
monthly = pd.DataFrame({
    sid: raw[sid].resample("MS").mean()
    for sid in ["MORTGAGE30US", "DGS10", "FEDFUNDS", "UNRATE", "CPIAUCSL"]
})
monthly["DRSFRMACBS"] = raw["DRSFRMACBS"].resample("MS").ffill(limit=2).reindex(monthly.index)
monthly = monthly.loc[START:]

## 3. Derived columns
- **Inflation_YoY** = year-over-year % change in CPI (`pct_change(12)*100`), computed on the full CPI history so 1991 has a value.
- **Mortgage_Spread** = 30 years mortgage rate minus 10 years Treasury yield.

In [4]:
cpi = raw["CPIAUCSL"].resample("MS").mean()
monthly["Inflation_YoY"] = (cpi.pct_change(12) * 100).reindex(monthly.index)
monthly["Mortgage_Spread"] = monthly["MORTGAGE30US"] - monthly["DGS10"]

## 4. Handle gaps
October 2025 has no UNRATE or CPI (BLS did not publish it), so that month is dropped rather than invented. The latest months of delinquency are blank because the series is published with a lag.

In [5]:
core = ["MORTGAGE30US", "FEDFUNDS", "DGS10", "UNRATE", "CPIAUCSL", "Inflation_YoY"]
monthly = monthly.dropna(subset=core).round(3)
monthly.index.name = "Date"

## 5. Save

In [6]:
monthly.reset_index().to_csv(DATA / "fred_clean.csv", index=False)
print(f"{len(monthly)} rows, {monthly.index.min():%Y-%m} -> {monthly.index.max():%Y-%m}")
print(monthly.isna().sum().to_string())
monthly.tail()

427 rows, 1991-01 -> 2026-08
MORTGAGE30US       0
DGS10              0
FEDFUNDS           0
UNRATE             0
CPIAUCSL           0
DRSFRMACBS         4
Inflation_YoY      0
Mortgage_Spread    0


,MORTGAGE30US,DGS10,FEDFUNDS,UNRATE,CPIAUCSL,DRSFRMACBS,Inflation_YoY,Mortgage_Spread
Date,,,,,,,,
2026-04-01,6.332,4.319,3.64,4.3,332.407,1.86,3.779,2.013
2026-05-01,6.442,4.484,3.63,4.3,333.979,NaN,4.167,1.958
2026-06-01,6.490,4.470,3.63,4.2,332.568,NaN,3.464,2.020
2026-07-01,6.542,4.600,3.63,4.1,332.813,NaN,3.304,1.942
2026-08-01,6.668,4.684,3.63,4.1,334.131,NaN,3.353,1.983
